# Convert PDF pages to JPEG images

This notebook uses `pypdfium2` to convert every PDF in an input folder into JPEG page images. 

Each PDF gets its own folder under the output folder, named after the PDF file (without `.pdf`). For example, `pdfs/report.pdf` is saved in `pdf_pages/report/` as `page_0001.jpg`, `page_0002.jpg`, and so on.

Put your PDFs in the input folder configured below, then run the notebook cells from top to bottom. The original PDFs are not modified.

In [6]:
%pip install pypdfium2 pillow

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 636.0 kB/s  0:00:05 eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [13]:
from pathlib import Path
import pypdfium2 as pdfium

# Change these folders to match where you keep your PDFs and converted images.
INPUT_DIR = Path("pdfs")
OUTPUT_DIR = Path("pdf_pages")
DPI = 300
JPEG_QUALITY = 90
OVERWRITE = False

if not INPUT_DIR.is_dir():
    raise FileNotFoundError(
        f"PDF input folder does not exist: {INPUT_DIR}. "
    )


pdf_paths = sorted(
    path for path in INPUT_DIR.iterdir()
    if path.is_file() and path.suffix.lower() == ".pdf"
)

print(f"Found {len(pdf_paths)} PDF file(s) in {INPUT_DIR}.")

Found 5 PDF file(s) in pdfs.


In [14]:
from contextlib import closing

total_saved = 0
total_skipped = 0

for pdf_path in pdf_paths:
    document_output_dir = OUTPUT_DIR / pdf_path.stem
    document_output_dir.mkdir(parents=True, exist_ok=True)

    saved = 0
    skipped = 0
    with closing(pdfium.PdfDocument(pdf_path)) as document:
        for page_index in range(len(document)):
            page_number = page_index + 1
            image_path = document_output_dir / f"page_{page_number:04d}.jpg"
            if image_path.exists() and not OVERWRITE:
                skipped += 1
                continue

            with closing(document[page_index]) as page:
                with closing(page.render(scale=DPI / 72)) as bitmap:
                    with closing(bitmap.to_pil()) as image:
                        image.save(
                            image_path,
                            format="JPEG",
                            quality=JPEG_QUALITY,
                            optimize=True,
                        )
            saved += 1

    total_saved += saved
    total_skipped += skipped
    print(
        f"{pdf_path.name}: saved {saved} page(s), skipped {skipped}; "
        f"folder: {document_output_dir}"
    )

print(
    f"\nFinished: {total_saved} page(s) saved, "
    f"{total_skipped} existing page image(s) skipped."
)

handwriting sample.pdf: saved 19 page(s), skipped 0; folder: pdf_pages/handwriting sample
scanned_1.pdf: saved 3 page(s), skipped 0; folder: pdf_pages/scanned_1
scanned_2.pdf: saved 2 page(s), skipped 0; folder: pdf_pages/scanned_2
scanned_3.pdf: saved 2 page(s), skipped 0; folder: pdf_pages/scanned_3
scanned_chinese.pdf: saved 2 page(s), skipped 0; folder: pdf_pages/scanned_chinese

Finished: 28 page(s) saved, 0 existing page image(s) skipped.
